<div style="background:#102A43;color:#ffffff;border-radius:11px;padding:30px 28px 24px;border-top:7px solid #18B7AC;">
<div style="color:#67DED3;letter-spacing:2px;font-weight:700;font-size:12px;">INTRODUCTION TO MACHINE LEARNING · GAZA SKY GEEKS</div>
<div style="font-size:31px;font-weight:750;line-height:1.25;margin:12px 0 9px;">Lecture 11 · Linear Regression:<br>Predicting Continuous Values</div>
<div style="font-size:15px;color:#DDEAF3;">Instructor: Diaeddin Rimawi &nbsp; | &nbsp; 21 November 2026 &nbsp; | &nbsp; Part 3: Supervised Learning</div>
<div style="margin-top:18px;padding-top:13px;border-top:1px solid #47667D;color:#D5F6F2;font-size:15px;">Our first trained Machine Learning model: from prepared observations to a learned prediction rule.</div>
</div>

### The question that begins Part 3

We can **clean** a dataset, **describe** its features, **compare** observations, and **transform** its representation.

But a prepared feature table does not yet answer a new kind of question:

> **Given the features of a house, can we estimate how much it will sell for?**

To answer, we need historical houses whose selling prices are already known. We will use these **labeled examples** to learn a numerical relationship—and then apply that relationship to a new house whose price is unknown.

**By the end, you should be able to:**

- Distinguish supervised learning from unsupervised learning, and regression from classification.
- Express a linear prediction as a slope, an intercept, and an input value.
- Explain **how a line is fitted** by minimizing squared residuals.
- Calculate a small least-squares model by hand and with NumPy before using scikit-learn.
- Train a real linear-regression model with `fit()` and make a prediction with `predict()`.
- Connect a **short training-only preprocessing pipeline** to a predictive estimator.


<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">01 · LECTURE 11</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">From prepared features to supervised learning</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">What is missing from our preprocessing pipeline?</div>
</div>

### What L10 taught us—and what it did *not* do

Last time, we learned to fit and apply transformations. For example, a `StandardScaler` fits on training features to learn a mean and a standard deviation; a PCA transformer fits to learn new directions.

Those are *learned transformations*, but neither gives a house-price prediction on its own.

**Supervised learning** introduces a second ingredient: a known target for each historical training observation.

- **Input features, `X`:** What we know about the house *before* its sale, such as area and neighborhood.
- **Target, `y`:** What we want to predict, here its historical sale price.
- **Labeled example, `(Xᵢ, yᵢ)`:** One historical house with its features and known price.
- **Model training:** Learn parameters connecting `X_train` to `y_train`.
- **Prediction:** Apply the learned parameters to a new `X_new`, without requiring its unknown price.

A preprocessing transformer often learns statistics from `X_train`. A regression estimator uses **both `X_train` and `y_train`** to learn predictive coefficients. The two operations share the method name `fit()` but learn different things.

### Two types of supervised prediction

| Question | Target `y` | Learning task |
|---|---|---|
| What price might this house sell for? | A numerical amount such as **$210,000** | **Regression** |
| Which penguin species is this? | A category such as **Adelie** | **Classification** |

Both are *supervised*: examples include the correct historical target. In contrast, methods such as the **PCA transformation from L10** can learn patterns in the feature values without target labels. We will study other unsupervised approaches later; no clustering algorithm is needed today.

**Important:** A numerical target tells us to investigate *regression methods*. It does **not** imply that every numerical relationship is linear.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">02 · LECTURE 11</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Which prediction problem are we solving?</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Choose the learning task before choosing its algorithm.</div>
</div>

### Decide before writing code

1. **What is the outcome?** The individual house's recorded selling price in US dollars.
2. **Do we have historical labels?** Yes: past transactions include sale prices.
3. **Is the outcome a category?** No. Price is numeric, so this is a **regression** task.
4. **What might explain price?** Living area, construction quality, construction year, and location are plausible inputs.
5. **What should we try first?** A **straight-line approximation** using only living area. Its parameters will be understandable and easy to inspect.

A straight line is a **first model, not a claim that house prices must follow a linear relationship**. Similar-sized houses can differ in quality, location, condition, and selling circumstances. A fitted line may be particularly unreliable for unusual sizes outside the training data.

### Meet our real dataset · Ames Housing

Our data describes **2,930 historical individual residential property sales** in Ames, Iowa (USA), between **2006 and 2010**, recorded in the Ames Housing dataset compiled by Dean De Cock. The original project-provided CSV has **82 columns**. For this lecture, we have kept **all 2,930 sales but only seven columns** so that we can focus on learning rather than extensive data preparation.

**One row = one sold house.** The house characteristics form the input features **`X`**; the recorded **`SalePrice`** is the known historical label **`y`**. We will use these pairs to learn a predictive relationship.

<div style="border:1px solid #BED7DD;border-radius:8px;padding:13px 14px;margin:12px 0;background:#F5FAFB;">
  <div style="font-weight:700;color:#102A43;margin-bottom:10px;font-size:13px;">HOW TO READ A LABELED HOUSING OBSERVATION</div>
  <div style="display:flex;gap:10px;align-items:stretch;flex-wrap:wrap;">
    <div style="border-radius:6px;background:#102A43;color:#FFFFFF;padding:10px 12px;flex:3;min-width:230px;">
      <div style="color:#65DFD4;font-weight:700;letter-spacing:.5px;">Features · X</div>
      <div style="font-size:13px;margin-top:6px;">Living area · Quality · Year built · Street frontage · Neighborhood</div>
    </div>
    <div style="align-self:center;font-size:22px;font-weight:700;color:#127F84;">→</div>
    <div style="border-radius:6px;background:#D9F1EE;color:#102A43;padding:10px 12px;flex:1;min-width:140px;">
      <div style="font-weight:700;">Target · y</div>
      <div style="font-size:13px;margin-top:6px;">Recorded sale price (USD)</div>
    </div>
  </div>
  <div style="font-size:12px;color:#465D65;margin-top:9px;">During training, the model sees both X and y. For a new house, it receives X and predicts an unknown y.</div>
</div>

| Column in our lecture file | Meaning | Modeling role |
|---|---|---|
| `GrLivArea` | Above-ground living area, **square feet** | First model's numerical feature |
| `OverallQual` | Overall material and finish quality, **ordinal 1–10 score** | Additional numerical input (treated as a score for this introductory model) |
| `YearBuilt` | Year the house was originally constructed | Additional numerical input |
| `LotFrontage` | Linear **feet** of street adjacent to the property; **490 missing values** in the complete file | Motivates median imputation |
| `Neighborhood` | Coded neighborhood within Ames | Motivates categorical encoding |
| `SalePrice` | **Recorded sale price of the individual house, USD** | **Regression target `y`** |
| `Id` | Observation number copied from original `Order`; not a house characteristic | Metadata, **never a predictor** |

We kept original values and missing cells, simply renaming several original headers (for example, `Gr Liv Area` → `GrLivArea`). The local file is `data/ames_housing_l11.csv`. It is bundled beside this notebook, so **no internet connection is needed**. More provenance and original-to-lecture column names are in `DATASET_README.md`.

This is **historical observational data**, not today's market prices :D

In [ ]:
# One familiar setup cell for today's entire notebook.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

NAVY, TEAL, GOLD, BLUE = "#102A43", "#18B7AC", "#DB9137", "#557AB5"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10.5,
                     "axes.spines.top": False, "axes.spines.right": False})
DATA_PATH = Path("data") / "ames_housing_l11.csv"
assert DATA_PATH.exists(), "Keep the notebook beside the bundled data/ folder."
houses = pd.read_csv(DATA_PATH)
EXPECTED_COLUMNS = ["Id", "GrLivArea", "OverallQual", "YearBuilt", "LotFrontage", "Neighborhood", "SalePrice"]
assert houses.shape == (2930, len(EXPECTED_COLUMNS))
assert houses.columns.tolist() == EXPECTED_COLUMNS
assert houses["Id"].is_unique and houses["SalePrice"].notna().all()
print(f"Loaded {len(houses):,} historical Ames house sales and {houses.shape[1]} selected columns.")
print("Our input features:", [c for c in houses if c not in ["Id", "SalePrice"]])
print("Our target: SalePrice (USD); metadata Id is excluded.")

### First protect our held-out examples

As we learned in L10, we split the dataset **before fitting any preprocessing statistics or predictive coefficients**. For this first modeling lecture, we use only **training** and **test** partitions.

- **Training set (80%):** used to explore relationships, compute preprocessing statistics, and learn regression coefficients.
- **Test set (20%):** put aside. We will **not inspect its features or labels, fit on it, report its errors, or use it to choose a model in L11**.

**Why no validation split yet?** We are learning how a first model fits. In **L12**, we will introduce a validation subset drawn **only from the current training portion**, keeping this test set untouched. That is when we learn how to compare models without making the final test observations part of our decisions.

`SalePrice` is our known target, and `Id` is metadata—not a predictor.

In [ ]:
FEATURES = ["GrLivArea", "OverallQual", "YearBuilt", "LotFrontage", "Neighborhood"]
X = houses[FEATURES].copy()
y = houses["SalePrice"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
assert len(X_train) + len(X_test) == len(houses)
assert set(X_train.index).isdisjoint(set(X_test.index))
print(f"Training houses: {len(X_train)}; untouched test houses: {len(X_test)}")
print("Missing training feature values:")
display(X_train.isna().sum().rename("Missing training values").to_frame())

# From this point onward, demonstrations inspect or fit only the TRAINING rows.
training_preview = X_train.head(5).copy()
training_preview["Actual sale price (USD)"] = y_train.loc[training_preview.index]
display(training_preview)

**Interpretation:** In the training data, `GrLivArea` has no missing values, so it is an ideal first feature for understanding a fitted line. The missing values in `LotFrontage` give us a **real reason** to impute values later, and `Neighborhood` gives us a reason to encode text categories. The preprocessing pipeline will learn everything from **training houses**.

### A quick visual introduction to our training houses

Before fitting a model, glance at how **living areas** and **historical prices** vary. These small distributions are **descriptive only**—they do not train a model. They use the training observations, not the held-out test set.

In [ ]:
# Two compact distributions help us understand the real houses in a few seconds.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(X_train["GrLivArea"], bins=28, color=TEAL,
             edgecolor="white", linewidth=.65)
axes[0].set(title="Training houses: living area", xlabel="Living area (square feet)", ylabel="Number of houses")
axes[1].hist(y_train / 1000, bins=28, color=NAVY,
             edgecolor="white", linewidth=.65)
axes[1].set(title="Training houses: sale prices", xlabel="Recorded price (thousand USD)", ylabel="Number of houses")
for ax in axes:
    ax.grid(axis="y", alpha=.13)
fig.suptitle("Dataset snapshot · two different numerical scales", fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

**What should we notice?** House sizes and prices differ widely, and a histogram tells us about each **individual feature's distribution**. It does **not** show whether they are related. For that, we next use a scatter plot with **one dot per sold training house**.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">03 · LECTURE 11</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Can one feature help us estimate price?</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Look at the training evidence before fitting a model.</div>
</div>

### Start with the question, not the API

Suppose we know only a house's living area. Would larger houses *tend* to sell for more?

A scatter plot shows one point per **training** house. We place living area on the horizontal axis and historical sale price on the vertical axis. We will express price in **thousands of dollars ($k)** on the plot for readability; the stored target remains dollars.

In [ ]:
fig, ax = plt.subplots(figsize=(9.8, 5.2))
ax.scatter(X_train["GrLivArea"], y_train / 1000,
           color=TEAL, s=22, alpha=.38, linewidth=0)
ax.set(title="Training houses: living area versus historical sale price",
       xlabel="Above-ground living area (square feet)",
       ylabel="Recorded sale price (thousand USD)")
ax.grid(alpha=.16)
plt.show()

**What can we read from this plot?** We can look for an overall direction, but the points will not lie exactly along a single line. The variation reminds us that **area is not the only factor** determining price.

At this stage we have *explored* a possible relationship. We have **not trained** anything yet. The next step is to express a possible prediction line as an equation.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">04 · LECTURE 11</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Turn a visible trend into a prediction rule</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">What do the slope and intercept actually mean?</div>
</div>

### The simplest regression model

$$\boxed{f(x) = ax + b ... OR ... \hat y = b_0 + b_1x}$$

- $x$: one input, here living area in **square feet**.
- $\hat y$: the **predicted** sale price; the hat means this is an estimate, not the actual observed price $y$.
- $b_0$: the **intercept**, where the prediction line crosses the vertical axis.
- $b_1$: the **slope**, the predicted change in price for one extra unit of $x$.

For example, if $b_1=125$ USD/sq ft, the line's price estimate rises by **\$125 for one additional square foot**, holding this one-feature model's assumption fixed. That does not mean adding a square foot *causes* an extra $125 in resale value.

An intercept is mathematically necessary for many models, but the price predicted for a **zero-square-foot house** need not have a sensible real-world meaning. It extrapolates beyond typical observed homes.

**Key idea:** In linear regression, the algorithm must **learn** numerical values of $b_0$ and $b_1$ from labeled examples.

### A small controlled example for the mathematics

Example: Use the four **illustrative (constructed)** house observations:

- $x$: **hundreds of square feet**, so $x=15$ means 1,500 sq ft.
- $y$: **thousands of US dollars**, so $y=213$ means $213,000.

| House | Living area (sq ft) | $x$ (hundreds sq ft) | Actual price $y$ ($k) |
|---|---:|---:|---:|
| A | 1,000 | 10 | 140 |
| B | 1,200 | 12 | 170 |
| C | 1,400 | 14 | 210 |
| D | 1,600 | 16 | 220 |

The units of the slope depend on the units of $x$ and $y$. We will always label them.

In [ ]:
# Two possible lines over the same four illustrative houses.
x_small = np.array([10., 12., 14., 16.])  # 100 square feet
price_small = np.array([140., 170., 210., 220.])  # thousand USD
x_curve = np.linspace(9.3, 16.7, 140)

fig, ax = plt.subplots(figsize=(9.3, 5.0))
ax.scatter(x_small, price_small, color=NAVY, s=76, zorder=4,
           label="Known house prices")
ax.plot(x_curve, 3 + 14*x_curve, color=TEAL, linewidth=2.7,
        label=r"Line A: $\hat y = 3 + 14x$")
ax.plot(x_curve, 40 + 10*x_curve, color=GOLD, linewidth=2.2,
        linestyle="--", label=r"Line B: $\hat y = 40 + 10x$")
ax.set(xlabel="Living area (hundreds of sq ft)",
       ylabel="Price (thousand USD)",
       title="Different slopes and intercepts give different predictions")
ax.legend(frameon=False)
ax.grid(alpha=.14)
plt.show()

**Interpretation:** Both equations can produce a prediction at any supplied $x$, but they do not predict the same prices. Choosing a line by eye is not a reliable fitting procedure. We need a rule that uses the **known training targets** to judge which line fits the observations better.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">05 · LECTURE 11</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">What does it mean to fit the line?</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Actual target → prediction → residual → squared residual.</div>
</div>

### From predictions to residuals

For observation $i$, a candidate line produces a prediction $\hat y_i$. The observed difference is its **residual**:

$$\boxed{e_i=y_i-\hat y_i}$$

- Positive residual: the observed price is **higher** than the prediction.
- Negative residual: the observed price is **lower** than the prediction.
- Zero residual: the line passes exactly through the observation.

If we simply add residuals, positive and negative differences can cancel. **Ordinary least squares (OLS)** therefore chooses coefficients that minimize the **sum of squared residuals**:

$$\boxed{SSE(b_0,b_1)=\sum_{i=1}^{n}(y_i-b_0-b_1x_i)^2}$$

Squaring prevents cancellation and gives larger residuals more weight. We are using **SSE as the fitting objective**, not yet teaching test-set error metrics; that evaluation belongs in L12.

In [ ]:
# Compare candidate lines using the SAME four observed target prices.
pred_a = 3 + 14*x_small
pred_b = 40 + 10*x_small
comparison = pd.DataFrame({
    "Area (100 sq ft)": x_small.astype(int),
    "Actual price ($k)": price_small.astype(int),
    "Line A ($k)": pred_a.astype(int),
    "Residual A ($k)": (price_small-pred_a).astype(int),
    "Line B ($k)": pred_b.astype(int),
    "Residual B ($k)": (price_small-pred_b).astype(int),
})
display(comparison)
sse_a = np.sum((price_small-pred_a)**2)
sse_b = np.sum((price_small-pred_b)**2)
print(f"Line A total squared residuals: {sse_a:.0f} ($k)^2")
print(f"Line B total squared residuals: {sse_b:.0f} ($k)^2")
assert sse_a == 180 and sse_b == 1400

In [ ]:
# Vertical segments show the residuals for Line A.
fig, ax = plt.subplots(figsize=(9.4, 5.0))
ax.plot(x_curve, 3+14*x_curve, color=TEAL, linewidth=2.7,
        label="Candidate line A")
ax.scatter(x_small, price_small, color=NAVY, s=68, zorder=4,
           label="Actual house prices")
ax.scatter(x_small, pred_a, color=GOLD, marker="x", s=66, zorder=5,
           label="Predicted house prices")
for x_i, actual, predicted in zip(x_small, price_small, pred_a):
    ax.vlines(x_i, min(actual,predicted), max(actual,predicted),
              color=BLUE, linewidth=2, linestyle="--")
    ax.annotate(f"{actual-predicted:+.0f}",
                (x_i+.10, (actual+predicted)/2), fontsize=9, color=BLUE)
ax.set(xlabel="Living area (hundreds of sq ft)",
       ylabel="Price (thousand USD)",
       title="A residual is the vertical actual-minus-predicted difference")
ax.legend(frameon=False, loc="upper left")
ax.grid(alpha=.15)
plt.show()

**Interpretation:** For these four observations, Line A gives smaller total squared residuals than Line B. But how could we find the **best** slope and intercept, instead of trying arbitrary lines? The fitting calculation provides the answer.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">06 · LECTURE 11</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Find the best-fitting line step by step</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Ordinary least squares for a single numerical feature.</div>
</div>

### Derive the parameters without advanced calculus

For ordinary least squares with one varying input feature, the slope and intercept can be calculated directly from the **training observations**:

$$\boxed{b_1=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}{\sum_i(x_i-\bar x)^2}}$$

$$\boxed{b_0=\bar y-b_1\bar x}$$

Here $\bar x$ and $\bar y$ are the means of the observed feature and target values.

- The **numerator** tells us whether $x$ and $y$ tend to vary together, and in which direction.
- The **denominator** measures how much the input values vary around their mean.
- Their ratio becomes the slope. The intercept then positions the line through the point $(\bar x,\bar y)$.

If every training $x$ is identical, the denominator is zero. There is not enough variation to determine a unique slope from this formula.

### Compute our four-house example

$$\bar x=\frac{10+12+14+16}{4}=13,\qquad\bar y=\frac{140+170+210+220}{4}=185.$$

The next table exposes **each intermediate quantity**. We are doing the fitting calculation, not estimating a performance score.

In [ ]:
mean_x, mean_y = x_small.mean(), price_small.mean()
calculation = pd.DataFrame({
    "x": x_small.astype(int),
    "y": price_small.astype(int),
    "x - mean(x)": (x_small-mean_x).astype(int),
    "y - mean(y)": (price_small-mean_y).astype(int),
    "product": ((x_small-mean_x)*(price_small-mean_y)).astype(int),
    "(x - mean(x))^2": ((x_small-mean_x)**2).astype(int),
})
display(calculation)
num = np.sum((x_small-mean_x)*(price_small-mean_y))
den = np.sum((x_small-mean_x)**2)
print(f"mean(x) = {mean_x:.0f}; mean(y) = {mean_y:.0f}")
print(f"Numerator = {num:.0f}; denominator = {den:.0f}")
print(f"Slope = {num:.0f}/{den:.0f} = {num/den:.0f} thousand USD per 100 sq ft")
assert num == 280 and den == 20

Therefore:

$$b_1=\frac{280}{20}=14,\qquad b_0=185-(14)(13)=3.$$

Our best-fitting line for these **four constructed observations** is:

$$\boxed{\hat y=3+14x}$$

Remember our units: this is **$3,000 + $14,000 for each 100 sq ft**. A house of 1,500 square feet has $x=15$:

$$\hat y=3+14(15)=213\text{ (thousand USD)}=\boxed{\$213{,}000}.$$

The observed prices need not lie exactly on the fitted line. OLS minimizes **total squared residuals**; it does not promise a perfect match for each training example.

In [ ]:
fitted_small = 3 + 14*x_small
worked = pd.DataFrame({
    "House": list("ABCD"),
    "Actual ($k)": price_small.astype(int),
    "OLS prediction ($k)": fitted_small.astype(int),
    "Residual ($k)": (price_small-fitted_small).astype(int),
    "Squared residual ($k)^2": ((price_small-fitted_small)**2).astype(int),
})
display(worked)
print("SSE for fitted line:", int(np.sum((price_small-fitted_small)**2)), "($k)^2")
print("Predicted price for 1,500 sq ft: $", f"{(3+14*15)*1000:,.0f}", sep="")
assert np.sum((price_small-fitted_small)**2) == 180

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">07 · LECTURE 11</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Describe the fitting algorithm</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">An understandable procedure before the library implementation.</div>
</div>

### Pseudocode: fit, then predict

We can now describe *exactly what the algorithm learns* without introducing derivatives, gradient descent, or matrix inversion.

```text
FIT_SIMPLE_LINEAR_REGRESSION(training_x, training_y):
    x_mean <- mean(training_x)
    y_mean <- mean(training_y)

    numerator   <- sum((x - x_mean) * (y - y_mean))
    denominator <- sum((x - x_mean)^2)
    if denominator == 0:
        report that slope cannot be uniquely determined

    slope     <- numerator / denominator
    intercept <- y_mean - slope * x_mean
    return intercept, slope

PREDICT(new_x, intercept, slope):
    return intercept + slope * new_x
```

Notice the separation: **`FIT` needs labeled historical examples**. Once the coefficients are learned, **`PREDICT` needs only a new input value and those coefficients**.

In [ ]:
# An original NumPy implementation of the one-feature algorithm.
def fit_simple_ols(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) != len(y) or len(x) < 2:
        raise ValueError("At least two paired observations are required.")
    x_mean, y_mean = x.mean(), y.mean()
    denominator = np.sum((x-x_mean)**2)
    if denominator == 0:
        raise ValueError("Slope is undefined: all feature values are identical.")
    slope = np.sum((x-x_mean)*(y-y_mean)) / denominator
    intercept = y_mean - slope*x_mean
    return intercept, slope

def simple_predict(x_new, intercept, slope):
    return intercept + slope*np.asarray(x_new)

manual_intercept, manual_slope = fit_simple_ols(x_small, price_small)
predicted_1500 = simple_predict(15, manual_intercept, manual_slope)
print(f"Learned intercept: {manual_intercept:.0f} ($k)")
print(f"Learned slope:     {manual_slope:.0f} ($k per 100 sq ft)")
print(f"Predicted price:   ${predicted_1500*1000:,.0f}")
np.testing.assert_allclose([manual_intercept, manual_slope, predicted_1500], [3,14,213])

**Interpretation:** We programmed a learning rule. The fitted values are **computed from observed examples**, not supplied as assumptions. If the training examples changed, the learned coefficients could change.

This is the important conceptual bridge to scikit-learn: the library automates the fitting computation, but it does not change what `fit` and `predict` mean.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">08 · LECTURE 11</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Train our first real predictive model</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Use scikit-learn only after understanding what fitting computes.</div>
</div>

### Fit a line to the **real Ames training houses**

We now return to our single real dataset. The first model uses only `GrLivArea`, measured in **square feet**; target price is measured in **dollars**. We deliberately **do not standardize** this input yet, so the learned slope can be interpreted directly as dollars per square foot.

In scikit-learn:

- `LinearRegression()` creates an **unfitted estimator**.
- `.fit(X_train, y_train)` learns `intercept_` and `coef_` using **training** examples.
- `.predict(X_new)` applies these learned values to a new input row.

A one-column DataFrame is used for `X` because scikit-learn expects a two-dimensional feature table, even when there is only one feature.

In [ ]:
X_train_area = X_train[["GrLivArea"]]  # Keep the 2D DataFrame shape.
area_model = LinearRegression()
area_model.fit(X_train_area, y_train)  # Training rows only.

real_intercept = float(area_model.intercept_)
real_slope = float(area_model.coef_[0])
print(f"Fitted line: predicted price = ${real_intercept:,.0f} + ${real_slope:,.2f} * living_area_sqft")
print(f"Slope interpretation: approximately ${real_slope:,.2f} per additional square foot")
print(f"Number of model coefficients: {len(area_model.coef_)}")
assert np.isfinite([real_intercept, real_slope]).all()

**What did `fit()` learn?** Only two predictive parameters for this one-feature model: one intercept and one slope. This is different from fitting a scaler, which learns a mean and a scale **without using the selling prices**.

We can check the learned line visually against its **training observations**. A straight line will not match every point perfectly, even though it was chosen to minimize the training sum of squared residuals.

In [ ]:
area_grid = np.linspace(X_train_area["GrLivArea"].min(),
                        X_train_area["GrLivArea"].max(), 180)
line_input = pd.DataFrame({"GrLivArea": area_grid})
line_prediction = area_model.predict(line_input)
fig, ax = plt.subplots(figsize=(9.7, 5.2))
ax.scatter(X_train_area["GrLivArea"], y_train/1000,
           color=TEAL, alpha=.32, s=20, linewidth=0,
           label="Historical TRAINING house sales")
ax.plot(area_grid, line_prediction/1000, color=NAVY, linewidth=2.7,
        label="OLS line learned from training labels")
ax.set(xlabel="Above-ground living area (sq ft)",
       ylabel="Recorded or predicted price ($k)",
       title="Our first trained model: one learned line through imperfect observations")
ax.legend(frameon=False)
ax.grid(alpha=.15)
plt.show()

### Predict an unobserved house price

Imagine a different house with 1,500 square feet of above-ground living area. We do **not** know its sale price. Supply **only the input feature**, using exactly the same column name as during fitting.

In [ ]:
new_house_area = pd.DataFrame({"GrLivArea": [1500]})
area_prediction = float(area_model.predict(new_house_area)[0])
print(f"For 1,500 sq ft, our one-feature model predicts about ${area_prediction:,.0f}.")

# A few TRAINING examples illustrate actual-versus-predicted values.
example_rows = X_train_area.head(4)
example_predictions = area_model.predict(example_rows)
examples = pd.DataFrame({
    "Living area (sq ft)": example_rows["GrLivArea"].to_numpy(),
    "Actual sale (USD)": y_train.loc[example_rows.index].to_numpy(),
    "Predicted sale (USD)": np.round(example_predictions).astype(int),
})
examples["Residual (USD)"] = examples["Actual sale (USD)"] - examples["Predicted sale (USD)"]
display(examples)
assert np.isfinite(area_prediction)

**Interpretation:** The difference between each training sale price and its fitted prediction is a residual. These examples **illustrate** prediction, not performance on unseen data. We are intentionally **not using the test set** or discussing evaluation scores today.

A one-feature estimate ignores much of what we know about houses. Let us add a few informative inputs and make the preprocessing steps *actually necessary*.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">09 · LECTURE 11</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Connect L10 preprocessing to predictive learning</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">One short pipeline: missing values + scaling + categorical encoding → regression.</div>
</div>

### What changes when we use more features?

We now want to include more of the known house information:

- **Numerical:** `GrLivArea`, `OverallQual`, `YearBuilt`, `LotFrontage`.
- **Categorical:** `Neighborhood`.

Two actual preparation problems matter:

1. `LotFrontage` contains **missing values**, and this regression estimator cannot use those missing values directly.
2. `Neighborhood` contains **categories written as text**, so we need numerical indicator columns.

We can now reuse L10 in **one short, purposeful pipeline**:

**Numerical features → training-median imputation → standardization**

**Neighborhood → one-hot encoding**

**Combined features → `LinearRegression`**

There are no missing `Neighborhood` values in this lecture's dataset, so there is **no need to add a categorical imputer**. Scaling is included as a familiar reusable preparation step, **not** because ordinary least squares inherently requires standardized features or because scaling guarantees better predictions. All learned preparation parameters and predictive coefficients come from the training split only.

In [ ]:
NUMERIC_FEATURES = ["GrLivArea", "OverallQual", "YearBuilt", "LotFrontage"]
CATEGORICAL_FEATURES = ["Neighborhood"]

numeric_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

prepare_houses = ColumnTransformer([
    ("numeric", numeric_steps, NUMERIC_FEATURES),
    ("neighborhood", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

# After preparation, the same predictive estimator can learn from more inputs.
house_pipeline = Pipeline([
    ("prepare", prepare_houses),
    ("model", LinearRegression()),
])
house_pipeline.fit(X_train, y_train)  # The whole pipeline is fitted on TRAINING only.

fitted_numeric_imputer = house_pipeline.named_steps["prepare"].named_transformers_["numeric"].named_steps["impute"]
fitted_encoder = house_pipeline.named_steps["prepare"].named_transformers_["neighborhood"]
print("Pipeline stages:", list(house_pipeline.named_steps))
print("Numeric medians learned from training data:")
display(pd.Series(fitted_numeric_imputer.statistics_, index=NUMERIC_FEATURES,
                  name="Train-only median").to_frame())
print(f"Neighborhood categories learned from training houses: {len(fitted_encoder.categories_[0])}")
print("New raw houses will be transformed and predicted using these learned parameters.")

### Make a prediction with the whole pipeline

Because the preprocessing steps and estimator live inside one `Pipeline`, `predict` can accept **raw, unscaled house features**. The pipeline will consistently perform the same transformations it learned at training time.

This new example intentionally has a **missing `LotFrontage`**, so median imputation is genuinely used. Its neighborhood label is supplied as text and encoded automatically.

In [ ]:
new_house_full = pd.DataFrame({
    "GrLivArea": [1500],
    "OverallQual": [7],
    "YearBuilt": [2001],
    "LotFrontage": [np.nan],  # The learned training median will be used.
    "Neighborhood": ["CollgCr"],
})
full_prediction = float(house_pipeline.predict(new_house_full)[0])
print(f"Area-only estimate for 1,500 sq ft: ${area_prediction:,.0f}")
print(f"Multifeature estimate for this hypothetical house: ${full_prediction:,.0f}")
print("The predictions differ because the models use different input information.")
print("This comparison is NOT evidence that either model generalizes better.")
assert np.isfinite(full_prediction)
assert np.all(np.isfinite(np.asarray(house_pipeline.predict(X_train.head(5)))))

### One workflow, two kinds of learned parameters

| Component | What `fit()` learns | What happens to a new raw house |
|---|---|---|
| `SimpleImputer` | **Training** medians for the numerical inputs | Fills missing numerical features |
| `StandardScaler` | **Training** means and scales | Uses them to standardize numerical inputs |
| `OneHotEncoder` | Neighborhood categories present in training | Converts neighborhood text to numerical indicators, ignoring previously unseen categories |
| **`LinearRegression`** | **Predictive intercept and coefficients from `(X_train, y_train)`** | **Combines the prepared features to predict a price** |

**Interpretation matters:** The multifeature pipeline's numerical coefficients use **standardized feature units**, so do not read its living-area coefficient as dollars per original square foot. Our first unscaled one-feature model was intentionally easier to interpret.

This targeted pipeline solves genuine problems in Ames Housing without repeating the preprocessing lectures. The test data remains untouched. Fitting a model is **not yet evidence of reliable prediction on unseen houses**.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">10 · LECTURE 11</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">What have we learned, and what comes next?</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">L11 teaches fitting and predicting; L12 will teach evaluating and comparing.</div>
</div>

### The entire story in one view

**1. Define the question:** Predict an individual house's sale price—**regression**, not classification.

**2. Identify the labeled examples:** `X` contains house features; `y` contains historically known sale prices.

**3. Protect unseen observations:** Split into **training** and **test** data. Fit nothing on the test set.

**4. Start simply:** A line $\hat y=b_0+b_1x$ turns living area into a numerical estimate.

**5. Explain how learning works:** The best-fitting one-feature line minimizes the **sum of squared training residuals**. Its slope and intercept are computed from the labeled examples.

**6. Train the model:** `fit()` learns coefficients; `predict()` uses those coefficients for unseen feature values.

**7. Make preprocessing reliable:** In a `Pipeline`, training-only imputation, scaling, and encoding can feed directly into a predictive estimator.

### Looking ahead · L12 on 24 November 2026

We now know **how to fit a line** and obtain predictions. But what does a good prediction look like—especially for an **unseen** house?

In L12 we will introduce a **validation set**, separate from our untouched test set, and learn to examine **MAE, MSE, RMSE, and $R^2$**. We will distinguish fitting training observations from generalizing beyond them, and investigate **underfitting, overfitting, model complexity, and polynomial regression**.

> **The question for L12:** A model can fit its training observations. How can we tell whether it predicts new observations reliably?



### Dataset and teaching sources

- Dean De Cock, *Ames, Iowa: Alternative to the Boston Housing Data as an End of Semester Regression Project*, *Journal of Statistics Education* (2011). **Original Ames Housing dataset: 2,930 sales and 82 variables.** This lecture uses all 2,930 observations from the CSV provided in the course project, retaining only seven documented fields. See `DATASET_README.md` for exact column mapping and origin.
- Ames dataset documentation: [OpenIntro Ames Housing](https://www.openintro.org/data/index.php?data=ames) and [AmesHousing project](https://github.com/topepo/AmesHousing). These are background descriptions; running this notebook requires no download.
- Prof. Antonio Liotta, predictive-analysis and linear/polynomial regression teaching notebooks have inspired this lecture material.
- scikit-learn documentation: [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), and [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html).
